# 03c Clean Country-Month-Year Feature Tables

This notebook applies the approved `config/source_cleaning_rules.csv` rules and writes cleaned feature outputs without modifying Silver.


In [ ]:
from pathlib import Path  # Work with filesystem paths.

REPO_URL = "https://github.com/Arobnett/HDX-sources-and-more-API-connection.git"  # Store the GitHub repository URL.
REPO_DIR = Path("/content/HDX-sources-and-more-API-connection")  # Define the local Colab clone folder.

if not REPO_DIR.exists():  # Clone the repo only when this runtime has not cloned it yet.
    !git clone {REPO_URL} {REPO_DIR}

%cd /content/HDX-sources-and-more-API-connection


In [ ]:
!git lfs install  # Enable Git LFS support in this Colab runtime.
!git lfs pull  # Download real large source files instead of pointer files.


In [ ]:
import sys  # Access Python's module search path.
from pathlib import Path  # Represent repository paths independently of operating system.

PROJECT_ROOT = Path.cwd()  # Treat the cloned repository as the project root.
SRC_DIR = PROJECT_ROOT / "src"  # Locate shared Python modules.
sys.path.insert(0, str(SRC_DIR))  # Make src modules importable in this Colab session.

from cleaning import clean_silver_directory  # Import the Step 03c cleaning runner.
from paths import CLEAN_DIR, CLEAN_REPORTS_DIR, SILVER_DIR, SOURCE_CLEANING_RULES_PATH, ensure_output_directories  # Import canonical paths.

print(f"Project root: {PROJECT_ROOT}")  # Show the active repository root.
print(f"Silver directory exists: {SILVER_DIR.exists()}")  # Confirm Silver inputs are available.
print(f"Rules file exists: {SOURCE_CLEANING_RULES_PATH.exists()}")  # Confirm the committed 03b rules are available.


In [ ]:
ensure_output_directories()  # Create generated output folders only.

summary_report, reject_rows = clean_silver_directory()  # Apply source-specific rules to every covered Silver source.

summary_report  # Display the validation summary.


In [ ]:
print(f"Cleaned output directory: {CLEAN_DIR}")  # Show where cleaned feature files were written.
for path in sorted(CLEAN_DIR.glob("*.csv")):  # List cleaned feature outputs.
    print(f"{path.name}: bytes={path.stat().st_size}")  # Print each output name and size.

print(f"Validation report: {CLEAN_REPORTS_DIR / 'cleaning_validation_report.csv'}")  # Show validation report path.
print(f"Reject rows report: {CLEAN_REPORTS_DIR / 'cleaning_reject_rows.csv'}")  # Show reject report path.


In [ ]:
errors = summary_report[summary_report["status"].eq("error")]  # Identify source-level cleaning failures.
bad_keys = summary_report[summary_report["key_unique"].eq(False)]  # Identify outputs that still violate country-month uniqueness.

print(f"Errors requiring resolution: {len(errors)}")  # Print source-level error count.
print(f"Outputs with duplicate country-month keys: {len(bad_keys)}")  # Print key validation failure count.

if len(errors) or len(bad_keys):  # Stop when cleaning failed validation.
    raise ValueError("03c cleaning validation failed; inspect summary_report.")  # Raise a visible notebook error.

print("03c smoke test passed: cleaned outputs were written and key validation passed.")  # Confirm successful run.
